In [7]:
import os
import subprocess
import time

# 1. Installeer zstd en Ollama (met forcering van het pad)
print("Bezig met installeren van dependencies...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Definieer het volledige pad naar ollama
# De installer zet hem meestal in /usr/local/bin/ollama
OLLAMA_PATH = "/usr/local/bin/ollama"

# 3. Start de server op de achtergrond
print("Ollama server opstarten...")
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15) # Geef de server tijd

# 4. Pull het model met het volledige pad
print("Model downloaden (kan even duren)...")
subprocess.run([OLLAMA_PATH, "pull", "llama3"])

# 4b. Pull het embedding model voor de PDF en GitHub tools
print("Embedding model downloaden...")
subprocess.run([OLLAMA_PATH, "pull", "nomic-embed-text"])

# 5. Controleer of het model er staat
print("\nGeïnstalleerde modellen:")
subprocess.run([OLLAMA_PATH, "list"])

# 6. CrewAI installeren (we negeren de errors van de Google-pakketten)
!pip install -q --no-warn-conflicts crewai langchain_community crewai_tools pymupdf

import crewai
import langchain_community
print("CrewAI is succesvol geladen!")

Bezig met installeren van dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree..

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                         
pulling 577073ffcc6c: 100% ▕██████████████████▏  110 B                         
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


Embedding model downloaden...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ 


Geïnstalleerde modellen:
NAME                       ID              SIZE      MODIFIED               
llama3:latest              365c0bd3c000    4.7 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    


pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


CrewAI is succesvol geladen!


In [13]:
!wget https://static-eu.marstekenergy.com/ems/resource/agreement/MarstekDeviceOpenApi.pdf

--2026-03-16 15:34:52--  https://static-eu.marstekenergy.com/ems/resource/agreement/MarstekDeviceOpenApi.pdf
Resolving static-eu.marstekenergy.com (static-eu.marstekenergy.com)... 18.244.202.86, 18.244.202.30, 18.244.202.71, ...
Connecting to static-eu.marstekenergy.com (static-eu.marstekenergy.com)|18.244.202.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 107952 (105K) [application/pdf]
Saving to: ‘MarstekDeviceOpenApi.pdf.1’

MarstekDeviceOpenAp 100%[===================>] 105.42K  --.-KB/s    in 0.02s   

2026-03-16 15:34:52 (4.91 MB/s) - ‘MarstekDeviceOpenApi.pdf.1’ saved [107952/107952]



In [18]:
import fitz  # PyMuPDF
from crewai.tools import tool

# --- CUSTOM LOKALE TOOLS ---

@tool("PDF_Lezer_Tool")
def pdf_lezer_tool(pdf_pad: str = '/kaggle/working/MarstekDeviceOpenApi.pdf'):
    """Leest de volledige inhoud van de Marstek API PDF. 
    Gebruik dit om byte-offsets en JSON velden te vinden."""
    try:
        doc = fitz.open(pdf_pad)
        tekst = ""
        for page in doc:
            tekst += page.get_text()
        return tekst
    except Exception as e:
        return f"Fout bij lezen PDF: {str(e)}"

@tool("Lokale_Bestand_Lezer")
def bestand_lezer(bestandsnaam: str):
    """Leest lokale Python of JSON bestanden uit de werkmap."""
    with open(bestandsnaam, 'r') as f:
        return f.read()

@tool("Bestand_Writer")
def bestand_writer(bestandsnaam: str, inhoud: str):
    """
    Schrijft of update een bestand in de werkmap (/kaggle/working/).
    Gebruik dit om de nieuwe sensor.py, coordinator.py of configuratiebestanden op te slaan.
    """
    try:
        # Zorg dat de map bestaat (bijv. als de agent schrijft naar 'custom_components/marstek/sensor.py')
        map_pad = os.path.dirname(bestandsnaam)
        if map_pad and not os.path.exists(map_pad):
            os.makedirs(map_pad)
            
        with open(bestandsnaam, 'w') as f:
            f.write(inhoud)
        return f"Succes: {bestandsnaam} is opgeslagen."
    except Exception as e:
        return f"Fout bij schrijven naar {bestandsnaam}: {e}"

In [16]:
!git clone -b testing https://github.com/Steavy/ha-marstek-local-api.git /kaggle/working/repo

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 709, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 709 (delta 205), reused 185 (delta 168), pack-reused 451 (from 1)
Receiving objects: 100% (709/709), 440.50 KiB | 10.24 MiB/s, done.
Resolving deltas: 100% (402/402), done.


In [17]:
import os
from crewai.tools import tool

# --- 2. DE GITHUB TOOL (Vervangt GithubSearchTool) ---
@tool("Repo_Bestand_Lezer")
def repo_lezer(bestandsnaam: str):
    """
    Leest een specifiek bestand uit de Marstek GitHub repo.
    Geef het pad op vanaf de root, bijv. 'custom_components/marstek/sensor.py'
    """
    base_path = "/kaggle/working/repo/"
    volledig_pad = os.path.join(base_path, bestandsnaam)
    try:
        with open(volledig_pad, 'r') as f:
            return f.read()
    except Exception as e:
        return f"Bestand {bestandsnaam} niet gevonden: {e}"

@tool("Repo_Lister")
def repo_lister(pad: str = "."):
    """Lijst alle bestanden in de repo op om te zien welke files beschikbaar zijn."""
    base_path = "/kaggle/working/repo/"
    target = os.path.join(base_path, pad)
    try:
        bestanden = []
        for root, dirs, files in os.walk(target):
            for file in files:
                bestanden.append(os.path.relpath(os.path.join(root, file), base_path))
        return "\n".join(bestanden)
    except Exception as e:
        return f"Fout bij lijsten van repo: {e}"

In [ ]:
import os
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import GithubSearchTool
from kaggle_secrets import UserSecretsClient

# 1. Gegevens ophalen
user_secrets = UserSecretsClient()
GH_TOKEN = user_secrets.get_secret("GH_MARSTEK_TOKEN")
GH_REPO = "Steavy/ha-marstek-local-api" # Alleen de repo naam voor de tool
OPENAI_API_KEY = "poep"

# 2. Centraal LLM object (Dit voorkomt OpenAI calls)
# We definiëren dit één keer voor zowel agents als tools.
local_llm = LLM(
    model="ollama/llama3",
    base_url="http://localhost:11434"
)

# 3. Tool Configuratie (Cruciaal voor de AuthenticationError)
# We dwingen zowel de LLM als de Embedder naar Ollama.
ollama_tool_config = {
    "llm": {
        "provider": "ollama",
        "config": {
            "model": "llama3",
            "base_url": "http://localhost:11434",
        }
    },
    "embedder": {
        "provider": "ollama",
        "config": {
            "model": "nomic-embed-text",
            "base_url": "http://localhost:11434",
        }
    }
}

# --- AGENTS ---
# We geven 'llm=local_llm' direct mee aan elke agent.
api_analyst = Agent(
    role='Marstek API Specialist',
    goal='Analyseer de PDF en vergelijk byte-offsets met de huidige GitHub code.',
    backstory='Expert in UDP protocollen en Marstek hardware.',
    tools=[pdf_lezer_tool, repo_lezer, repo_lister],
    llm=local_llm,
    verbose=True
)

developer = Agent(
    role='Home Assistant Developer',
    goal='Schrijf nieuwe Python code voor sensor.py en coordinator.py.',
    backstory='Senior Python dev die asynchrone integraties bouwt.',
    tools=[bestand_lezer, bestand_writer],
    llm=local_llm,
    verbose=True
)

# --- TASKS ---
task_scan = Task(
    description='Scan de PDF voor Passive mode en DOD parameters. Vergelijk met de repo.',
    agent=api_analyst,
    expected_output='Rapport met nieuwe JSON velden en byte-offsets.'
)

task_code = Task(
    description='Update de Python bestanden op basis van het rapport. Sla ze op in de werkmap.',
    agent=developer,
    context=[task_scan],
    expected_output='Gereviseerde Python code opgeslagen als bestanden.'
)

# --- CREW ---
marstek_crew = Crew(
    agents=[api_analyst, developer],
    tasks=[task_scan, task_code],
    process=Process.sequential,
    verbose=True
)

# Start de actie
result = marstek_crew.kickoff()

print("######################")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 42edb031-6c8c-4eeb-84fa-bcf295620200                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Scan de PDF voor Passive mode en DOD parameters. Vergelijk met de repo.                                  │
│  ID: 0a7c7a31-487f-416e-b4ab-5ef4f2834fa5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marstek API Specialist                                                                                  │
│                                                                                                                 │
│  Task: Scan de PDF voor Passive mode en DOD parameters. Vergelijk met de repo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pdf_lezer_tool                                                                                           │
│  Args: {"properties": {"pdf_pad": "/kaggle/working/MarstekDeviceOpenApi.pdf"}}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: pdf_lezer_tool                                                                                           │
│  Output: Property                                                                                               │
│  Type                                                                                                           │
│  Description                                                                                                    │
│  id                                                                                                             │
│  number                                                                                                         │
│  or string                                                                                                      │
│  An identifier established by the Client.                                                                       │
│  method                                                                                                         │
│  string                                                                                                         │
│  A Structured value that holds the parameter values to be used                                                  │
│  during the invocation of the method.                                                                           │
│  params                                                                                                         │
│  object                                                                                                         │
│  Parameters that the method takes.                                                                              │
│  Marstek Device Open API（Rev 2.0）                                                                             │
│                                                                                                                 │
│                   The Local API is provided “as is” for local use only.Use at your own risk. Marstek is not     │
│  liable                                                                                                         │
│  for any damages, data loss, or legal issues caused by your use of the API.You are responsible for              │
│  lawful and appropriate use.                                                                                    │
│  Ⅰ. Preface：                                                                                                   │
│                                                                                                                 │
│  Welcome!                                                                                                       │
│                   This document provides an introduction to the Open API for Marstek devices, which is          │
│  available to device owners and enables integration with third-party systems.                                   │
│                   While Marstek offers an official mobile app and cloud services, this Open API is designed     │
│  for                                                                                                            │
│  advanced users who wish to gain greater control over their devices and seamlessly integrate                    │
│  them into other management platforms.                                                                          │
│  Ⅱ. General Description                                                                                         │
│                                                           

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Property                                                                                                       │
│  Type                                                                                                           │
│  Description                                                                                                    │
│  id                                                                                                             │
│  number                                                                                                         │
│  or string                                                                                                      │
│  An identifier established by the Client.                                                                       │
│  method                                                                                                         │
│  string                                                                                                         │
│  A Structured value that holds the parameter values to be used                                                  │
│  during the invocation of the method.                                                                           │
│  params                                                                                                         │
│  object                                                                                                         │
│  Parameters that the method takes.                                                                              │
│  Marstek Device Open API（Rev 2.0）                                                                             │
│                                                                                                                 │
│                   The Local API is provided “as is” for local use only.Use at your own risk. Marstek is not     │
│  liable                                                                                                         │
│  for any damages, data loss, or legal issues caused by your use of the API.You are responsible for              │
│  lawful and appropriate use.                                                                                    │
│  Ⅰ. Preface：                                                                                                   │
│                                                                                                                 │
│  Welcome!                                                                                                       │
│                   This document provides an introduction to the Open API for Marstek devices, which is          │
│  available to device owners and enables integration with third-party systems.                                   │
│                   While Marstek offers an official mobile app and cloud services, this Open API is designed     │
│  for                                                                                                            │
│  advanced users who wish to gain greater control over their devices and seamlessly integrate                    │
│  them into other management platforms.                                                                          │
│  Ⅱ. General Description                                                                                         │
│                                                                                                                 │
│  Marstek devices communicate with third-party systems over a Local Area Network (LAN). Before                   │
│  using this API, please ensure that:                     